# 4b · Retrieve & ground ARIA's answers

**Core · Live-code · ~20 min**

Now we use the vector store from 4a to answer questions **from the manuals**, with citations.
This is the payoff of RAG: reliable, fact-based answers instead of confident guesses.

To keep things simple, we'll use the ready-made `RagIndex` helper (which does the chunk +
embed + store steps from 4a for us). Then we'll build the "grounding" prompt by hand so you
see exactly how retrieval and generation fit together.

Solution: [`solution/04b_retrieve_ground.ipynb`](solution/04b_retrieve_ground.ipynb).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from shared.rag import RagIndex
from shared.llm import chat

# Build the index: this chunks + embeds all five manuals (takes a few seconds).
index = RagIndex.from_manuals()
print(f"Indexed {len(index.chunks)} chunks from the manuals.")

## Step 1 — Retrieve: what does the store return?

Before generating any answer, let's *see* the retrieval step on its own. `index.retrieve(q, k)`
embeds the question, compares it to every stored chunk, and returns the `k` closest ones.

Run this and confirm the returned passages really are about oxygen / the scrubber. Retrieval
quality is everything in RAG — if the wrong passages come back, the answer will be wrong too.

In [ ]:
for hit in index.retrieve("What do I do if O2 drops in Module B?", k=3):
    print(f"--- from {hit['source']} ---")
    print(hit["text"][:200], "...\n")

## Step 2 — Ground: answer using only the retrieved text

This is the heart of RAG. We take the retrieved passages, paste them into the prompt as
**context**, and give the model a strict instruction: *answer only from these excerpts, cite
the source file, and if the answer isn't there, say so.*

That instruction is what stops hallucination — the model is no longer free-associating from
memory; it's constrained to the facts we handed it. Write ARIA's grounding **system prompt**
in the TODO below.

In [ ]:
def grounded_answer(query, k=3):
    hits = index.retrieve(query, k=k)
    # Join the retrieved passages into one context block, each tagged with its source.
    context = "\n\n---\n\n".join(f"[{h['source']}]\n{h['text']}" for h in hits)

    # TODO: write a system prompt that tells ARIA to:
    #   - answer ONLY using the provided excerpts,
    #   - cite the source file in square brackets,
    #   - say it doesn't have the information if it's not in the excerpts.
    system = "Replace this with the grounding instruction described above."

    return chat(f"Excerpts:\n{context}\n\nQuestion: {query}", system=system, temperature=0.2)

print(grounded_answer("What do I do if O2 drops in Module B?"))

## Step 3 — Hallucination, gone

Remember the part number ARIA cheerfully invented in Module 3? Ask again — but now it's
grounded. Because that fact appears in *none* of the retrieved manual passages, a
well-behaved grounded ARIA should now admit it doesn't have that information. Compare this to
the confident fabrication from before: that's the whole value of RAG in one contrast.

In [ ]:
print(grounded_answer("What is the exact part number of the Module B scrubber cartridge?"))

# For convenience, shared/rag.py has this same behaviour built in as index.answer(...).
# We'll rely on it in Modules 5 and 7:
print("\n--- via the shared helper ---")
print(index.answer("How should we respond to a dust storm?"))

### 🌱 Stretch (optional)

See [`stretch_evaluation.ipynb`](stretch_evaluation.ipynb) to *measure* how good retrieval is
on a small set of known questions — moving from "seems fine" to an actual score.

## ✅ Recap

You retrieved the right manual passages and forced ARIA to answer from them, with citations —
turning a confident guesser into a trustworthy, fact-grounded assistant. RAG is the pattern
you'll reach for any time an AI needs to answer from *your* specific documents.

# 4b · Retrieve & ground ARIA's answers

**Core · Live-code · ~20 min**

Use the vector store to answer questions **from the manuals**, with citations.

> Solution: [`solution/04b_retrieve_ground.ipynb`](solution/04b_retrieve_ground.ipynb).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from shared.rag import RagIndex
from shared.llm import chat

# Build the index (chunks + embeds all manuals). Takes a few seconds.
index = RagIndex.from_manuals()
print(f"Indexed {len(index.chunks)} chunks")

## Step 1 — retrieve

What passages does the store return for our emergency question?

In [ ]:
for hit in index.retrieve("What do I do if O2 drops in Module B?", k=3):
    print(f"--- {hit['source']} ---")
    print(hit["text"][:200], "...\n")

## Step 2 — ground the answer

Now build a prompt that includes those passages and instructs ARIA to answer **only** from
them, citing the source. Try writing it yourself before using the helper.

In [ ]:
def grounded_answer(query, k=3):
    hits = index.retrieve(query, k=k)
    context = "\n\n---\n\n".join(f"[{h['source']}]\n{h['text']}" for h in hits)
    # TODO: write a system prompt: answer ONLY from the excerpts, cite the source file,
    #       and if the answer isn't present, say you don't have that information.
    system = "# TODO"
    return chat(f"Excerpts:\n{context}\n\nQuestion: {query}", system=system, temperature=0.2)

print(grounded_answer("What do I do if O2 drops in Module B?"))

## Step 3 — hallucination, gone

Re-ask the part-number question from Module 3. A grounded ARIA should now admit it doesn't
know, instead of inventing an answer.

In [ ]:
print(grounded_answer("What is the exact part number of the Module B scrubber cartridge?"))

# Compare with the built-in helper (shared/rag.py) used in later modules:
print("\n--- via shared.rag ---")
print(index.answer("How should we respond to a dust storm?"))

### 🌱 Stretch
See [`stretch_evaluation.ipynb`](stretch_evaluation.ipynb) to measure how good retrieval is.